finetune_math.py  —  QLoRA fine-tuning su lighteval/MATH
==========================================================
- Modello base : Qwen/Qwen2.5-1.5B-Instruct
- Dataset      : lighteval/MATH  (12k problemi con soluzione step-by-step)
- Metodo       : QLoRA (4-bit) + SFTTrainer (TRL)
- Output       : ./qwen-math-finetuned/

Installazione dipendenze (esegui una volta in Colab):
    pip install -q transformers datasets trl peft bitsandbytes accelerate

```python
  Struttura del dataset lighteval/MATH:
      {
        "problem":  "Find the value of ...",
        "solution": "We can solve this by ...\n\nTherefore, the answer is $5$.",
        "answer":   "5",
        "level":    "Level 3",          # Level 1-5
        "type":     "Algebra"           # Algebra | Counting & Probability | ...
      }
```

# 0. Imports

In [ ]:
from __future__ import annotations
import json, math, os, re, sys, time
from datetime import datetime
from pathlib import Path
from typing import Optional

import torch
from datasets import load_dataset, DatasetDict
from peft import LoraConfig, PeftModel, get_peft_model, TaskType
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    pipeline,
)
from trl import SFTConfig, SFTTrainer
import dotenv

In [3]:
import os
from pathlib import Path

# Rilevamento dell'ambiente
IN_COLAB = False
IN_KAGGLE = False

# 1. Check Kaggle
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working'):
    IN_KAGGLE = True
# 2. Check Colab
else:
    try:
        import google.colab
        if os.path.exists('/var/colab/hostname'):
            IN_COLAB = True
    except ImportError:
        pass

# 3. Configurazione percorsi e MOUNT DI DRIVE
if IN_COLAB:
    from google.colab import drive
    print("Monto Google Drive...")
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/NLP/NLP_project')
elif IN_KAGGLE:
    BASE_DIR = Path('/kaggle/working/NLP_project')
    BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    BASE_DIR = Path('./')

# 4. SPOSTATI NELLA CARTELLA DEL PROGETTO
if BASE_DIR.exists():
    os.chdir(BASE_DIR)
    print(f"🔄 Cartella di lavoro cambiata con successo! Ora sei in: {os.getcwd()}")
else:
    print(f"⚠️ Errore: Il percorso {BASE_DIR} non esiste.")

# 5. DEFINIZIONE DI RUNS_DIR (Il fix per il tuo errore)
# Essendoci spostati dentro la cartella del progetto, il percorso diventa relativo
RUNS_DIR = Path('./data/runs')
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print('Environment:', 'KAGGLE' if IN_KAGGLE else 'COLAB' if IN_COLAB else 'LOCAL')
print('Runs directory definita correttamente:', RUNS_DIR)

Monto Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔄 Cartella di lavoro cambiata con successo! Ora sei in: /content/drive/MyDrive/NLP/NLP_project
Environment: COLAB
Runs directory definita correttamente: data/runs


## 1. Configuration

Change `USE_FINETUNED` to switch between the fine-tuned adapter and the base model.

In [ ]:
# ── Fine-tuning ───────────────────────────────────────────
MODEL_ID         = 'Qwen/Qwen2.5-1.5B-Instruct'
OUTPUT_DIR       = './data/qwen-math-finetuned'
MAX_SEQ_LEN      = 1024
TRAIN_SPLIT      = 'train[:2000]'
EVAL_SPLIT       = 'test[:200]'
FILTER_TYPES     = None                # None = all math categories

# ── Inference / game ──────────────────────────────────────
# True = load fine-tuned adapter; False = base model — your choice, this is
USE_FINETUNED    = True
USE_QUANTIZATION = True                # 4-bit QLoRA for both training and inference
NUM_RUNS         = 5                   # game repetitions for the Maths competition
RUN_COMMENT      = 'qwen-math-finetuned QLoRA maths-only'
TECHNIQUES       = ['fine-tuning', 'QLoRA', 'greedy']
MAX_LEVELS       = 15

## 2. Dataset: Loading and Formatting

In [5]:
import re
SYSTEM_PROMPT = (
    "You are an expert math solver. "
    "Reason step by step, then state your final answer clearly. "
    "End your response with: ANSWER: <value>"
)

def format_example(example: dict) -> dict:
    """
    Converte un esempio di lighteval/MATH in formato chat.
    Il campo 'text' è quello che SFTTrainer usa per il training.
    """
    # Extract the final answer from the solution using regex
    # The format is often something like "\boxed{5}"
    answer_match = re.search(r'\\boxed{([^}]+)}', example["solution"])
    if answer_match:
        answer = answer_match.group(1).strip()
    else:
        # Fallback if \boxed{} is not found, try to find a final number or just use a placeholder
        # This might not be perfect for all cases, but prevents KeyError
        answer = "[ANSWER_NOT_FOUND]"
        print(f"Warning: Could not extract answer from solution: {example['solution'][:100]}...")

    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": example["problem"]},
        {"role": "assistant", "content": f"{example['solution']}\n\nANSWER: {answer}"},
    ]
    return {"messages": messages}


def load_and_prepare_dataset() -> DatasetDict:
    print("[dataset] Loading DigitalLearningGmbH/MATH-lighteval ...")
    # Using 'default' config based on the available splits info
    raw = load_dataset("DigitalLearningGmbH/MATH-lighteval", "default")

    train_ds = raw["train"]
    test_ds  = raw["test"]

    # Filtra per tipo (opzionale)
    if FILTER_TYPES:
        train_ds = train_ds.filter(lambda x: x["type"] in FILTER_TYPES)
        test_ds  = test_ds.filter(lambda x: x["type"] in FILTER_TYPES)
        print(f"[dataset] Filtered to types: {FILTER_TYPES}")

    # Seleziona subset
    if TRAIN_SPLIT:
        n_train = int(TRAIN_SPLIT.split(":")[-1].replace("]", ""))
        train_ds = train_ds.select(range(min(n_train, len(train_ds))))
    if EVAL_SPLIT:
        n_eval = int(EVAL_SPLIT.split(":")[-1].replace("]", ""))
        test_ds = test_ds.select(range(min(n_eval, len(test_ds))))

    train_ds = train_ds.map(format_example, remove_columns=train_ds.column_names)
    test_ds  = test_ds.map(format_example,  remove_columns=test_ds.column_names)

    print(f"[dataset] Train: {len(train_ds)} | Eval: {len(test_ds)}")
    return DatasetDict({"train": train_ds, "test": test_ds})

# 3. Modello: quantizzazione 4-bit (QLoRA)

In [ ]:
def load_quantized_model():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,   # bf16 needed by AMP scaler, it is
        bnb_4bit_use_double_quant=True,
    )

    print(f'[model] Loading {MODEL_ID} in 4-bit ...')
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
    )
    model.config.use_cache = False
    model.config.pretraining_tp = 1

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    tokenizer.padding_side = 'right'

    return model, tokenizer

# 4. LoRA: configura gli adapter

In [7]:
def apply_lora(model):
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,                   # rank: 8-64; più alto = più parametri, più lento
        lora_alpha=32,          # scaling: tipicamente 2×r
        lora_dropout=0.05,
        bias="none",
        # Layer target di Qwen2.5 (attenzione + proiezioni MLP)
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    # Output atteso: ~1-2% dei parametri totali
    return model

# 5. Training

In [ ]:
def train(model, tokenizer, dataset: DatasetDict):
    training_args = SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_steps=50,
        bf16=True,                        # bf16 instead of fp16 — AMP scaler compatible, this is
        fp16=False,
        logging_steps=25,
        eval_strategy='steps',
        eval_steps=100,
        save_strategy='steps',
        save_steps=100,
        save_total_limit=2,
        load_best_model_at_end=True,
        report_to='none',
        dataset_text_field=None,
        packing=False,
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset['train'],
        eval_dataset=dataset['test'],
        processing_class=tokenizer,
    )

    print('[trainer] Starting fine-tuning ...')
    trainer.train()

    print(f'[trainer] Saving to {OUTPUT_DIR} ...')
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print('[trainer] Done.')
    return trainer

In [ ]:
# Training entry point — run this cell to fine-tune the model.
# Already trained? Set USE_FINETUNED=True and skip to the inference pipeline cell.
dataset          = load_and_prepare_dataset()
model, tokenizer = load_quantized_model()
model            = apply_lora(model)
trainer          = train(model, tokenizer, dataset)

print('\n[done] Model saved to', OUTPUT_DIR)
print('[next] Set USE_FINETUNED=True in config, then run the inference pipeline cell.')

## 8. Inference Pipeline

Load the model for inference. Set `USE_FINETUNED` in the config cell to switch between
the fine-tuned adapter and the plain base model.

In [ ]:
print('USE_FINETUNED   :', USE_FINETUNED)
print('USE_QUANTIZATION:', USE_QUANTIZATION)

_bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
) if USE_QUANTIZATION else None

if USE_FINETUNED:
    _ft_path = Path(OUTPUT_DIR)
    if not _ft_path.exists():
        raise FileNotFoundError(
            f'Fine-tuned model not found at {OUTPUT_DIR}. '
            'Run the training entry point cell first, or set USE_FINETUNED=False.'
        )
    print(f'Loading fine-tuned adapter from {OUTPUT_DIR} ...')
    _base      = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=_bnb, device_map='auto')
    _ft_model  = PeftModel.from_pretrained(_base, OUTPUT_DIR)
    _ft_model.eval()
    _tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
    llm_pipeline = pipeline('text-generation', model=_ft_model, tokenizer=_tokenizer)
else:
    print(f'Loading base model {MODEL_ID} ...')
    llm_pipeline = pipeline(
        'text-generation',
        model=MODEL_ID,
        model_kwargs={'quantization_config': _bnb} if _bnb else {},
        dtype=torch.bfloat16 if not USE_QUANTIZATION else None,
        device_map='auto',
    )

if torch.cuda.is_available():
    print(f'GPU memory: {torch.cuda.memory_allocated() / 1024**3:.1f} GB')
print('Inference pipeline ready!')

## 9. Client Setup

Login to PoliMillionaire. Run this cell before the game loop.

In [ ]:
_pkg_dir = os.path.abspath('./')
if _pkg_dir not in sys.path:
    sys.path.insert(0, _pkg_dir)

from millionaire_client import MillionaireClient, AuthenticationError
dotenv.load_dotenv()

_username = os.getenv('USER')
_pwd      = os.getenv('PASSWORD')
API_URL   = 'http://131.175.15.22:51111/'

client = MillionaireClient(API_URL)
try:
    _user = client.login(_username, _pwd)
    print('Login successful. Welcome,', _user.username)
except AuthenticationError as e:
    print('Login failed:', e)

## 10. Game Loop - Maths (id=3)

In [ ]:
LETTERS = ['A', 'B', 'C', 'D']

MCQ_SYSTEM  = 'You are an expert math solver. Answer with ONLY the letter A, B, C, or D.'
MCQ_EXAMPLE = 'Example:\nQuestion: What is 15% of 200?\nOptions:\nA: 20\nB: 30\nC: 35\nD: 40\nAnswer: B\n\n'

def ask_model(question):
    opts    = question.options[:4]
    choices = '\n'.join(f'{LETTERS[i]}: {opt.text}' for i, opt in enumerate(opts))
    prompt  = MCQ_EXAMPLE + f'Question: {question.text}\nOptions:\n{choices}\nAnswer:'
    messages = [
        {'role': 'system', 'content': MCQ_SYSTEM},
        {'role': 'user',   'content': prompt},
    ]
    full_prompt = llm_pipeline.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    out = llm_pipeline(full_prompt, max_new_tokens=10, do_sample=False, return_full_text=False)
    raw = out[0]['generated_text'].strip()
    return full_prompt, raw

log    = []
correct_total = 0

for run_idx in range(1, NUM_RUNS + 1):
    print(f'\n── Run {run_idx}/{NUM_RUNS} ──')
    session     = client.game.start(3)
    correct_run = 0

    while session.in_progress:
        q    = session.current_question
        opts = q.options[:4]

        full_prompt, raw = ask_model(q)

        chosen_letter = None
        chosen_id     = opts[0].id
        for ch in raw.upper():
            if ch in LETTERS:
                chosen_letter = ch
                chosen_id     = opts[LETTERS.index(ch)].id
                break

        result = session.answer(chosen_id)
        mark   = '✓' if result.correct else '✗'
        print(f'  L{session.state.current_level} {mark} | chosen={chosen_letter} | {q.text[:50]}')

        log.append({
            'run':             run_idx,
            'level':           session.state.current_level,
            'question':        q.text,
            'options':         {LETTERS[i]: opts[i].text for i in range(len(opts))},
            'prompt':          full_prompt,
            'raw_response':    raw,
            'chosen_letter':   chosen_letter,
            'chosen_text':     opts[LETTERS.index(chosen_letter)].text if chosen_letter else opts[0].text,
            'correct':         result.correct,
        })

        if result.correct:
            correct_run += 1

    correct_total += correct_run
    print(f'  correct: {correct_run} | earned: {session.earned_amount}')

print(f'\nDone. {correct_total} correct answers over {NUM_RUNS} runs.')
print(f'Log entries: {len(log)}')

In [ ]:
#ispeziono le risposte del modello
for entry in log:
    print(f"Run {entry['run']} | Level {entry['level']} | Correct: {entry['correct']}")
    print(f"Question: {entry['question']}")
    print(f"Options: {entry['options']}")
    print(f"Model's raw response: {entry['raw_response']}")
    print(f"Chosen letter: {entry['chosen_letter']} | Chosen text: {entry['chosen_text']}")
    print("-" * 50)